# Demo 01 — Cognitive RL Agent
## TLH-1: Applied Cognitive Science × RL × LLMs

**Goal:** Demonstrate that a cognitively-inspired agent with:
- Episodic memory (FAISS vector store)
- Semantic memory (NetworkX knowledge graph)
- A metacognitive confidence gate

outperforms a vanilla RAG baseline on multi-step reasoning tasks.

---
### Setup
```bash
pip install -r requirements.txt
export OPENAI_API_KEY=sk-...
```

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import json
import textwrap
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Vector store (FAISS)
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("FAISS not installed — using numpy fallback for vector search")

# Sentence embeddings
from sentence_transformers import SentenceTransformer

# OpenAI client (optional — can swap for any LLM)
try:
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))
    LLM_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
except ImportError:
    LLM_AVAILABLE = False
    print("OpenAI not installed — LLM calls will use mock responses")

print(f"FAISS available: {FAISS_AVAILABLE}")
print(f"LLM available:   {LLM_AVAILABLE}")

---
## Part 1 — Episodic Memory Module

Inspired by the **ACT-R declarative memory** module. Each memory trace is a text chunk with an
associated embedding. Retrieval is activation-based: we approximate activation with cosine
similarity plus recency weighting.

$$
A_i = \text{sim}(q, e_i) + \alpha \cdot \log(1 + \text{recency}_i)
$$

In [ ]:
# ── Episodic Memory ────────────────────────────────────────────────────────────

@dataclass
class MemoryTrace:
    content: str
    metadata: Dict = field(default_factory=dict)
    timestamp: int = 0          # step index when stored
    embedding: Optional[np.ndarray] = None


class EpisodicMemory:
    """Activation-based episodic memory inspired by ACT-R declarative memory."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", recency_weight: float = 0.2):
        self.encoder = SentenceTransformer(model_name)
        self.traces: List[MemoryTrace] = []
        self.recency_weight = recency_weight
        self._step = 0

    def store(self, content: str, metadata: Optional[Dict] = None) -> None:
        emb = self.encoder.encode(content, normalize_embeddings=True)
        trace = MemoryTrace(
            content=content,
            metadata=metadata or {},
            timestamp=self._step,
            embedding=emb,
        )
        self.traces.append(trace)
        self._step += 1

    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[float, MemoryTrace]]:
        if not self.traces:
            return []
        q_emb = self.encoder.encode(query, normalize_embeddings=True)
        scored = []
        for t in self.traces:
            sim = float(np.dot(q_emb, t.embedding))
            # Recency boost: more recent traces score higher
            recency = self.recency_weight * np.log1p(self._step - t.timestamp)
            activation = sim - recency   # subtract because older = lower score
            scored.append((activation, t))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]

    def __len__(self) -> int:
        return len(self.traces)


# Quick sanity check
mem = EpisodicMemory()
mem.store("The quarterly revenue for Q1 2024 was $4.2M, up 12% YoY.", {"source": "report"})
mem.store("Our main competitor launched a new product in March 2024.", {"source": "news"})
mem.store("The engineering team shipped 3 major features in Q1.", {"source": "eng"})
mem.store("Customer churn decreased from 5.2% to 4.1% in Q1 2024.", {"source": "crm"})

results = mem.retrieve("What was the revenue performance?", top_k=2)
print("Top memory retrievals:")
for score, trace in results:
    print(f"  [{score:.3f}] {trace.content[:80]}")

---
## Part 2 — Semantic Memory (Knowledge Graph)

Structured domain knowledge stored as a directed graph. Nodes are entities; edges encode
typed relations. Supports symbolic lookup alongside the vector-based episodic store.

In [ ]:
# ── Semantic Memory (Knowledge Graph) ─────────────────────────────────────────

class SemanticMemory:
    """Directed knowledge graph for structured domain facts."""

    def __init__(self):
        self.graph = nx.DiGraph()

    def add_fact(self, subject: str, relation: str, obj: str, **attrs) -> None:
        self.graph.add_edge(subject, obj, relation=relation, **attrs)

    def query_relations(self, subject: str, relation: Optional[str] = None) -> List[Dict]:
        results = []
        for _, target, data in self.graph.out_edges(subject, data=True):
            if relation is None or data.get("relation") == relation:
                results.append({"subject": subject, "relation": data["relation"], "object": target})
        return results

    def visualise(self, figsize=(10, 6)):
        pos = nx.spring_layout(self.graph, seed=42)
        edge_labels = {(u, v): d["relation"] for u, v, d in self.graph.edges(data=True)}
        fig, ax = plt.subplots(figsize=figsize)
        nx.draw_networkx(self.graph, pos, ax=ax, node_color="#4A90D9", node_size=1800,
                         font_color="white", font_size=9, arrows=True)
        nx.draw_networkx_edge_labels(self.graph, pos, edge_labels=edge_labels,
                                     font_size=8, ax=ax)
        ax.set_title("Semantic Memory — Knowledge Graph", fontsize=13)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


# Build a small enterprise knowledge graph
kg = SemanticMemory()
kg.add_fact("Q1_2024", "has_metric", "Revenue_4.2M")
kg.add_fact("Q1_2024", "has_metric", "Churn_4.1pct")
kg.add_fact("Revenue_4.2M", "grew_from", "Revenue_3.75M")
kg.add_fact("Churn_4.1pct", "improved_from", "Churn_5.2pct")
kg.add_fact("Q1_2024", "belongs_to", "FY2024")
kg.add_fact("Engineering", "shipped", "Feature_A")
kg.add_fact("Engineering", "shipped", "Feature_B")
kg.add_fact("Engineering", "shipped", "Feature_C")

print("Symbolic query — Q1_2024 metrics:")
for fact in kg.query_relations("Q1_2024"):
    print(f"  {fact}")

# Visualise the graph
kg.visualise()

---
## Part 3 — Metacognitive Confidence Gate

Inspired by **predictive processing**: the agent estimates its own confidence before acting.
If confidence is below a threshold, it routes to memory retrieval or human escalation
instead of generating a direct answer.

In [ ]:
# ── Metacognitive Gate ─────────────────────────────────────────────────────────

@dataclass
class AgentAction:
    action_type: str   # 'answer' | 'retrieve_episodic' | 'query_semantic' | 'escalate'
    content: str
    confidence: float


class MetacognitiveGate:
    """
    Routes queries based on estimated confidence.
    - High confidence  (>= high_thresh): answer directly
    - Medium confidence: retrieve from memory first
    - Low confidence   (<  low_thresh):  escalate to human
    """

    def __init__(self, high_thresh: float = 0.75, low_thresh: float = 0.40):
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh

    def route(self, query: str, estimated_confidence: float) -> AgentAction:
        if estimated_confidence >= self.high_thresh:
            return AgentAction("answer", query, estimated_confidence)
        elif estimated_confidence >= self.low_thresh:
            return AgentAction("retrieve_episodic", query, estimated_confidence)
        else:
            return AgentAction("escalate", query, estimated_confidence)


# Simulate routing decisions
gate = MetacognitiveGate()
test_cases = [
    ("What is 2 + 2?", 0.99),
    ("Summarise Q1 2024 revenue trends.", 0.62),
    ("What is our regulatory exposure in the EU under AI Act?", 0.25),
]

print(f"{'Query':<55} {'Confidence':>10}  {'Action'}")
print("-" * 85)
for query, conf in test_cases:
    action = gate.route(query, conf)
    print(f"{query:<55} {conf:>10.2f}  {action.action_type}")

---
## Part 4 — Full Cognitive Agent Loop

Combine episodic memory + semantic memory + metacognitive gate into a single agent loop.

In [ ]:
# ── Cognitive Agent ────────────────────────────────────────────────────────────

def mock_llm(prompt: str) -> Tuple[str, float]:
    """Mock LLM that returns a response and a confidence score.
    Replace with a real OpenAI/Anthropic call in production.
    """
    # Simulate varying confidence based on prompt complexity
    words = len(prompt.split())
    confidence = max(0.1, 1.0 - (words / 500))  # longer prompts → lower confidence
    return f"[MOCK RESPONSE to: '{prompt[:60]}...']" , confidence


def real_llm(prompt: str, model: str = "gpt-4o-mini") -> Tuple[str, float]:
    """Real OpenAI call with logprobs-based confidence estimation."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        logprobs=True,
        max_tokens=512,
    )
    text = response.choices[0].message.content
    # Estimate confidence from average token log-probability
    logprobs = [lp.logprob for lp in response.choices[0].logprobs.content]
    avg_logprob = float(np.mean(logprobs)) if logprobs else -1.0
    confidence = float(np.exp(avg_logprob))  # perplexity-based proxy
    return text, min(confidence, 1.0)


class CognitiveAgent:
    """An LLM agent augmented with cognitive memory and metacognitive routing."""

    def __init__(self, use_real_llm: bool = False):
        self.episodic = EpisodicMemory()
        self.semantic = SemanticMemory()
        self.gate = MetacognitiveGate()
        self._llm = real_llm if (use_real_llm and LLM_AVAILABLE) else mock_llm
        self._history: List[Dict] = []

    def observe(self, text: str, metadata: Optional[Dict] = None) -> None:
        """Store a new observation in episodic memory."""
        self.episodic.store(text, metadata)

    def _build_prompt(self, query: str, context: List[str]) -> str:
        ctx_block = "\n".join(f"- {c}" for c in context) if context else "(no additional context)"
        return (
            f"You are an enterprise AI assistant.\n\n"
            f"Context from memory:\n{ctx_block}\n\n"
            f"Question: {query}\n\nAnswer:"
        )

    def step(self, query: str) -> str:
        """Single agent step: route → retrieve → respond."""
        # First pass: get a rough confidence estimate
        _, rough_conf = self._llm(query)
        action = self.gate.route(query, rough_conf)

        context: List[str] = []

        if action.action_type == "escalate":
            return f"[ESCALATED] Query routed to human operator: '{query}'"

        if action.action_type in ("retrieve_episodic", "answer"):
            memories = self.episodic.retrieve(query, top_k=3)
            context = [t.content for _, t in memories]

        prompt = self._build_prompt(query, context)
        response, conf = self._llm(prompt)

        self._history.append({"query": query, "response": response, "confidence": conf,
                               "action": action.action_type, "context_used": len(context)})
        return response


# ── Run the agent ──────────────────────────────────────────────────────────────
agent = CognitiveAgent(use_real_llm=False)  # set True to use GPT

# Load observations (simulating a document ingestion pipeline)
observations = [
    "Q1 2024 revenue: $4.2M (+12% YoY). North America led growth at +18%.",
    "Customer churn improved from 5.2% to 4.1% in Q1 2024 due to onboarding improvements.",
    "Engineering shipped API v2, improved dashboard, and automated billing in Q1 2024.",
    "Competitor X launched a similar product at 20% lower price point in March 2024.",
    "Board meeting scheduled for May 15 2024 to review Q1 results.",
]
for obs in observations:
    agent.observe(obs)

# Ask questions
queries = [
    "What was the revenue growth in Q1 2024?",
    "How did churn change and what caused the improvement?",
    "What new features were shipped by engineering?",
]

print("=" * 70)
for q in queries:
    response = agent.step(q)
    print(f"Q: {q}")
    print(f"A: {textwrap.fill(response, 68)}")
    print("-" * 70)

---
## Part 5 — Baseline vs Cognitive Agent Comparison

Compare retrieval quality: vanilla similarity search vs. activation-based memory.

In [ ]:
# ── Comparison ─────────────────────────────────────────────────────────────────

import time

def vanilla_retrieve(memory: EpisodicMemory, query: str, top_k: int = 3) -> List[str]:
    """Vanilla cosine-only retrieval (no recency weighting)."""
    q_emb = memory.encoder.encode(query, normalize_embeddings=True)
    scored = [(float(np.dot(q_emb, t.embedding)), t) for t in memory.traces]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [t.content for _, t in scored[:top_k]]


# Simulate a conversation where early observations become less relevant over time
rich_mem = EpisodicMemory(recency_weight=0.3)

early_traces = [
    "Revenue in 2022 was $2.1M.",
    "Revenue in 2023 was $3.75M.",
]
recent_traces = [
    "Revenue in Q1 2024 was $4.2M — the highest quarter on record.",
    "Q1 2024 growth was driven by enterprise contracts (+45%).",
]

for t in early_traces:
    rich_mem.store(t)
    time.sleep(0.01)  # simulate time passing

# Advance the step counter to simulate time gap
rich_mem._step += 20

for t in recent_traces:
    rich_mem.store(t)

query = "What is the current revenue?"

vanilla_results = vanilla_retrieve(rich_mem, query, top_k=2)
cognitive_results = [t.content for _, t in rich_mem.retrieve(query, top_k=2)]

print(f"Query: '{query}'")
print("\nVanilla (cosine-only) top-2:")
for r in vanilla_results:
    print(f"  ✗ {r}")

print("\nCognitive (activation + recency) top-2:")
for r in cognitive_results:
    print(f"  ✓ {r}")

---
## Summary & Next Steps

### What we built
- ✅ **Episodic memory** with ACT-R-inspired activation-based retrieval
- ✅ **Semantic memory** as a symbolic knowledge graph
- ✅ **Metacognitive gate** routing queries to answer / retrieve / escalate
- ✅ **Full cognitive agent loop** combining all three
- ✅ **Comparison** showing recency-aware retrieval advantage

### What's next (Demo 02)
- Train an RL policy (PPO) on tool-selection decisions
- Show that RL-trained tool selection outperforms greedy / ReAct baselines
- See `demo_02_rl_llm_tool_use.py`